# Security ML 

### I. Basic CIFAR training with MLP (no obfuscation) 

Model can be configured differently to obtain better results. This is just a barebones example.

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [27]:
# normalization function 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# load data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, drop_last=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, drop_last=True)

np.random.seed(42)
sample_X, sample_Y = next(iter(trainloader))
batch_size = sample_X.shape[0]
d0 = sample_X.flatten(start_dim=1).shape[1]
n = sample_X.flatten(start_dim=1).shape[0]
c = len(torch.unique(sample_Y))

In [28]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [30]:
# initialize model, loss fn, optimization scheme
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device) # verify if using gpu

model = MLP(input_dim=d0, num_classes=c).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

cuda


In [31]:
# train
NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    model.train()
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        X = inputs.flatten(start_dim=1)
        Y = torch.eye(c, device=device)[labels]
        optimizer.zero_grad()
        loss = criterion(model(X), Y)
        loss.backward()
        optimizer.step()
    
    # Eval
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            X = images.flatten(start_dim=1)
            predicted = model(X).argmax(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    print(f'Epoch {epoch}: {100 * correct / total:.2f}%')

Epoch 0: 45.83%
Epoch 1: 48.33%
Epoch 2: 50.69%
Epoch 3: 51.55%
Epoch 4: 50.78%
Epoch 5: 52.23%
Epoch 6: 52.02%
Epoch 7: 52.82%
Epoch 8: 52.71%
Epoch 9: 52.94%


### II. With obfuscation

In [ ]:
d = 3072 # flattened size of X (i.e. d=d0)
m = batch_size # m = batchsize for simplicity

W = torch.randn(d0, d, device=device)

M = torch.rand(m, n, device=device) + 1e-8 # + 1e-8 to keep positive
M = M / M.sum(dim=1, keepdim=True) # row sums = 1

perm1 = torch.randperm(m, device=device)
Pi_1 = torch.zeros(m, m, device=device)
Pi_1[torch.arange(m), perm1] = 1

perm2 = torch.randperm(c, device=device)
Pi_2 = torch.zeros(c, c, device=device)
Pi_2[torch.arange(c), perm2] = 1

B = torch.rand(m, d, device=device)

In [33]:
class ObfuscatedMLP(nn.Module):
    def __init__(self, input_dim, num_classes):                                                                                                                                                                       
        super(ObfuscatedMLP, self).__init__()                                                                                                                                                                         
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_obfuscated = ObfuscatedMLP(input_dim=d, num_classes=c).to(device)
optimizer = optim.Adam(model_obfuscated.parameters(), lr=0.001)
criterion = nn.MSELoss()

# criterion = nn.KLDivLoss(reduction='batchmean')
# model output needs log_softmax:
# loss = criterion(F.log_softmax(model_obfuscated(X), dim=1), Y)

In [34]:
# train
NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    model_obfuscated.train()
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        # obfuscate X
        X = images.flatten(start_dim=1) @ W
        X = M @ X
        X = Pi_1 @ X
        X = X + B

        # obfuscate Y
        Y = torch.eye(c, device=device)[labels] # make one-hot
        Y = Y @ Pi_2

        optimizer.zero_grad()
        loss = criterion(model_obfuscated(X), Y)
        loss.backward()
        optimizer.step()
    
    # eval
    model_obfuscated.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            # Project test data with same W (no mixing/permutation for eval)
            X_test = images.flatten(start_dim=1) @ W

            # Get predictions and invert class permutation
            preds = model_obfuscated(X_test) @ Pi_2.T  # invert Pi_2
            predicted = preds.argmax(1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    print(f'Epoch {epoch}: {100 * correct / total:.2f}%')

Epoch 0: 11.04%
Epoch 1: 10.71%
Epoch 2: 10.40%
Epoch 3: 10.54%
Epoch 4: 15.55%
Epoch 5: 13.96%
Epoch 6: 14.83%
Epoch 7: 13.30%
Epoch 8: 13.82%
Epoch 9: 7.11%


### Misc (i.e. throwaway code)

In [ ]:
# matrix masking
def W(X, d):
    X = X.flatten(start_dim=1)
    d0 = X.shape[1] # number of columns
    W = torch.randn(d0, d, device=X.device, dtype=X.dtype) # Gaussian Matrix
    return X @ W # n x d

# data mixing
def M(m, X):
    # assumes flattened X (n x d)
    n = X.shape[0] # number of rows
    M = torch.rand(m, n, device=X.device, dtype=X.dtype) + 1e-8 # add 1e-8 to make it strictly positive
    M = M / M.sum(dim=1, keepdim=True) # row sums = 1
    return M @ X # m x d

# X permutation
def Pi_1(X):
    m = X.shape[0]
    perm = torch.randperm(m, device=X.device)
    Pi = torch.zeros(m, m, device=X.device, dtype=X.dtype)
    Pi[torch.arange(m), perm] = 1
    return Pi @ X # m x d

# Y permutation
def Pi_2(Y):
    c = Y.shape[1] # number of columns
    perm = torch.randperm(c, device=Y.device)
    Pi = torch.zeros(c, c, device=Y.device, dtype=Y.dtype)
    Pi[torch.arange(c), perm] = 1
    return Y @ Pi, Pi # n x c, also returns Pi_2 for inverse later

# noise
def B(m, d, X):
    return X + torch.rand(m, d, device=X.device, dtype=X.dtype)